# Colab T4 실측 노트북 (중간고사 숫자표용)

**런타임 → 런타임 유형 변경 → T4 GPU**를 고른 뒤 **런타임 → 모두 실행**만 하면 된다. 전부 끝나면 **파일 → 다운로드 → .ipynb 다운로드**로 받는다.

| 부분 | 내용 | 걸리는 시간 (대략) |
|---|---|---|
| 1 | 과제1 프로그래밍 B 설정 11개 (각 5에폭, N_TRAIN 500은 30에폭도) | 약 12분 |
| 2 | 과제1 프로그래밍 A YOLO 설정 10개의 추론 시간 | 약 1분 |
| 3 | 중간고사 템플릿 설정 4개 (Pet 이진·3클래스, VOC 사람 가중치 유무, VOC 내려받기 2GB 포함) | 약 6분 |

각 부분 끝에 표가 출력된다. 중간에 끊겨도 앞부분 표는 남는다. 맨 위 `RUN_PART3 = False`로 바꾸면 3부(VOC)를 건너뛴다.

In [ ]:
RUN_PART1, RUN_PART2, RUN_PART3 = True, True, True
QUICK = False   # True면 에폭을 1로 줄여 빠르게 동작만 확인

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import time, random, copy, json
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print("device:", device, "|", GPU_NAME, "| torch", torch.__version__)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

In [ ]:
# 과제1 B와 같은 U-Net (out_ch=1이면 sigmoid, 아니면 로짓)
def double_conv(i_, o_, bn=True):
    L = []
    for i, o in [(i_, o_), (o_, o_)]:
        L.append(nn.Conv2d(i, o, 3, padding=1))
        if bn: L.append(nn.BatchNorm2d(o))
        L.append(nn.ReLU(inplace=True))
    return nn.Sequential(*L)

class UNet(nn.Module):
    def __init__(self, base=16, n_pool=3, skip=True, bn=True, out_ch=1):
        super().__init__(); self.skip = skip; self.out_ch = out_ch
        chs = [base * 2 ** k for k in range(n_pool + 1)]
        self.encoders = nn.ModuleList([double_conv(3, chs[0], bn)] + [double_conv(chs[k-1], chs[k], bn) for k in range(1, n_pool + 1)])
        self.pool = nn.MaxPool2d(2)
        self.decoders = nn.ModuleList([double_conv(chs[k+1] + (chs[k] if skip else 0), chs[k], bn) for k in reversed(range(n_pool))])
        self.final = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        skips = []
        for k, e in enumerate(self.encoders):
            if k > 0: x = self.pool(x)
            x = e(x); skips.append(x)
        x = skips.pop()
        for d in self.decoders:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            s = skips.pop()
            x = d(torch.cat([s, x], 1) if self.skip else x)
        out = self.final(x)
        return torch.sigmoid(out) if self.out_ch == 1 else out

def n_params(m):
    return sum(p.numel() for p in m.parameters())

def infer_ms(model, size):
    model.eval(); x = torch.rand(1, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(10): model(x)
        sync(); t = time.time()
        for _ in range(50): model(x)
        sync()
    return (time.time() - t) / 50 * 1000

## 1부 · 과제1 프로그래밍 B 설정 (과제1 노트북과 같은 데이터 처리·학습)

In [ ]:
if RUN_PART1:
    raw_train = datasets.OxfordIIITPet("data", split="trainval", target_types="segmentation", download=True)
    raw_test  = datasets.OxfordIIITPet("data", split="test",     target_types="segmentation", download=True)

    class PetSegDataset(Dataset):
        def __init__(self, raw, n, size):
            n = len(raw) if n is None else min(n, len(raw))
            self.images, self.masks = [], []
            for i in range(n):
                img, tri = raw[i]
                img = img.convert("RGB").resize((size, size), Image.BILINEAR); tri = tri.resize((size, size), Image.NEAREST)
                self.images.append(np.array(img, dtype=np.float32) / 255.0); self.masks.append((np.array(tri) == 1).astype(np.float32))
        def __len__(self): return len(self.images)
        def __getitem__(self, i): return torch.tensor(self.images[i]).permute(2, 0, 1), torch.tensor(self.masks[i])

    def iou_score(p, t, th=0.5, eps=1e-6):
        p = (p > th).float()
        return (((p * t).sum((1, 2)) + eps) / ((((p + t) > 0).float().sum((1, 2))) + eps)).mean().item()

    DEF = dict(IMG_SIZE=128, N_TRAIN=None, N_TEST=500, BATCH_SIZE=16, BASE_CH=16, N_POOL=3, USE_SKIP=True, USE_BN=True, LR=1e-3, EPOCHS=5)
    EXPS = [("기본", {}), ("풀링 2번", {"N_POOL": 2}), ("풀링 4번", {"N_POOL": 4}), ("스킵 Off", {"USE_SKIP": False}),
            ("BASE_CH 8", {"BASE_CH": 8}), ("BASE_CH 32", {"BASE_CH": 32}), ("LR 1e-1", {"LR": 1e-1}), ("LR 1e-5", {"LR": 1e-5}),
            ("IMG_SIZE 64", {"IMG_SIZE": 64}), ("N_TRAIN 500", {"N_TRAIN": 500}), ("N_TRAIN 500, 30에폭", {"N_TRAIN": 500, "EPOCHS": 30})]
    cache, rows1 = {}, []
    for name, ch in EXPS:
        cfg = {**DEF, **ch}
        if QUICK: cfg["EPOCHS"] = 1
        torch.manual_seed(0); random.seed(0); np.random.seed(0)
        key = (cfg["IMG_SIZE"], cfg["N_TRAIN"])
        if key not in cache:
            t = time.time()
            cache[key] = (PetSegDataset(raw_train, cfg["N_TRAIN"], cfg["IMG_SIZE"]), PetSegDataset(raw_test, cfg["N_TEST"], cfg["IMG_SIZE"]))
            print(f"전처리 {key}: {time.time() - t:.0f}초")
        tr, te = cache[key]
        tl = DataLoader(tr, batch_size=cfg["BATCH_SIZE"], shuffle=True); vl = DataLoader(te, batch_size=cfg["BATCH_SIZE"])
        model = UNet(cfg["BASE_CH"], cfg["N_POOL"], cfg["USE_SKIP"], cfg["USE_BN"]).to(device)
        crit = nn.BCELoss(); opt = torch.optim.Adam(model.parameters(), lr=cfg["LR"])
        tl_hist, te_hist, iou_hist, sec_hist = [], [], [], []
        for ep in range(cfg["EPOCHS"]):
            model.train(); tot = 0; sync(); t = time.time()
            for x, y in tl:
                x, y = x.to(device), y.to(device)
                opt.zero_grad(); p = model(x).squeeze(1); loss = crit(p, y); loss.backward(); opt.step(); tot += loss.item() * len(x)
            sync(); sec_hist.append(time.time() - t)
            model.eval(); L = I = n = 0
            with torch.no_grad():
                for x, y in vl:
                    x, y = x.to(device), y.to(device); p = model(x).squeeze(1)
                    L += crit(p, y).item() * len(x); I += iou_score(p, y) * len(x); n += len(x)
            tl_hist.append(tot / len(tr)); te_hist.append(L / n); iou_hist.append(I / n)
        rows1.append({"설정": name, "파라미터 수": n_params(model), "에폭당 학습 시간(초)": round(float(np.mean(sec_hist)), 1),
                      "1에폭 train loss": round(tl_hist[0], 4), "마지막 train loss": round(tl_hist[-1], 4), "마지막 test loss": round(te_hist[-1], 4),
                      "최종 test IoU": round(iou_hist[-1], 3), "최고 test IoU(에폭)": f"{max(iou_hist):.3f} ({int(np.argmax(iou_hist)) + 1})",
                      "에폭별 IoU": " ".join(f"{v:.3f}" for v in iou_hist[:10]) + (" ..." if len(iou_hist) > 10 else ""),
                      "장당 추론(ms)": round(infer_ms(model, cfg["IMG_SIZE"]), 2)})
        print(f"[{name}] IoU {iou_hist[-1]:.3f}, 에폭당 {np.mean(sec_hist):.1f}초")
    df1 = pd.DataFrame(rows1); print(f"\n=== 1부 결과 ({GPU_NAME}) ==="); display(df1)

## 2부 · 과제1 프로그래밍 A YOLO 설정 (bus.jpg, 추론 시간은 같은 호출 5번 평균)

In [ ]:
if RUN_PART2:
    from ultralytics import YOLO
    from collections import Counter
    import requests
    from io import BytesIO
    img = Image.open(BytesIO(requests.get("https://ultralytics.com/images/bus.jpg", timeout=10).content)).convert("RGB")
    models = {}
    def run(model_name="yolov8n-seg.pt", conf=0.25, imgsz=640):
        if model_name not in models: models[model_name] = YOLO(model_name)
        m = models[model_name]
        for _ in range(3): m.predict(img, conf=conf, imgsz=imgsz, device=0 if device.type == "cuda" else "cpu", verbose=False)
        ts = []
        for _ in range(5):
            r = m.predict(img, conf=conf, imgsz=imgsz, device=0 if device.type == "cuda" else "cpu", verbose=False)[0]; ts.append(r.speed["inference"])
        names = [m.names[int(c)] for c in r.boxes.cls]
        return {"모델": model_name, "conf": conf, "imgsz": imgsz, "파라미터 수": sum(p.numel() for p in m.model.parameters()),
                "탐지 개수": len(r.boxes), "평균 점수": round(float(r.boxes.conf.mean()), 3) if len(r.boxes) else None,
                "추론 시간(ms)": round(float(np.mean(ts)), 1), "클래스": dict(Counter(names))}
    rows2 = [run(conf=c) for c in [0.05, 0.25, 0.5, 0.8]] + [run(imgsz=s) for s in [320, 640, 1280]] + \
            [run(model_name=n) for n in ["yolov8n-seg.pt", "yolov8s-seg.pt", "yolov8m-seg.pt"]]
    df2 = pd.DataFrame(rows2); print(f"=== 2부 결과 ({GPU_NAME}) ==="); display(df2)

## 3부 · 중간고사 템플릿 설정 (템플릿과 같은 데이터 처리: 좌우 뒤집기, VOC 255 무시, 대상 없는 사진은 오탐으로 따로)

In [ ]:
if RUN_PART3:
    IGNORE = 255
    def label_fn(option, task, m, pet_pos=(1,), voc_target=(15,)):
        m = np.array(m).astype(np.int64)
        if option == "pet":
            return np.isin(m, pet_pos).astype(np.int64) if task == "binary" else m - 1
        out = np.where(np.isin(m, voc_target), 1, 0) if task == "binary" else m.copy()
        out[m == 255] = IGNORE
        return out

    class SegDataset(Dataset):
        def __init__(self, raw, option, task, n=None, size=128, augment=False):
            n = len(raw) if n is None else min(n, len(raw)); self.augment = augment
            self.images, self.masks = [], []
            for i in range(n):
                img, mask = raw[i]
                self.images.append(np.array(img.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8))
                self.masks.append(label_fn(option, task, mask.resize((size, size), Image.NEAREST)).astype(np.uint8))
        def __len__(self): return len(self.images)
        def __getitem__(self, i):
            x = torch.tensor(self.images[i], dtype=torch.float32).permute(2, 0, 1) / 255.0
            y = torch.tensor(self.masks[i], dtype=torch.long)
            if self.augment and random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            return x, y

    def run_template(name, option, task, epochs=10, pos_weight=None, class_weight=None):
        torch.manual_seed(0); random.seed(0); np.random.seed(0)
        if option == "pet":
            rtr = datasets.OxfordIIITPet("data", split="trainval", target_types="segmentation", download=True)
            rte = datasets.OxfordIIITPet("data", split="test", target_types="segmentation", download=True)
            K = 1 if task == "binary" else 3
        else:
            rtr = datasets.VOCSegmentation("data", year="2012", image_set="train", download=True)
            rte = datasets.VOCSegmentation("data", year="2012", image_set="val", download=True)
            K = 1 if task == "binary" else 21
        tr = SegDataset(rtr, option, task, augment=True); te = SegDataset(rte, option, task, n=500)
        masks = np.stack(tr.masks); valid = masks != IGNORE
        info = {}
        if task == "binary":
            pos = float((masks == 1).sum() / valid.sum()); info["대상 픽셀 비율"] = f"{pos:.1%}"
            info["대상 없는 사진 비율"] = f"{np.mean([(m == 1).sum() == 0 for m in tr.masks]):.1%}"
            pw = np.sqrt((1 - pos) / pos) if pos_weight == "auto" else pos_weight
            bce = nn.BCELoss(reduction="none")
            def crit(out, t):
                v = (t != IGNORE).float(); tt = t.clamp(max=1).float()
                w = v if pw is None else v * torch.where(tt == 1, torch.tensor(float(pw), device=device), torch.tensor(1.0, device=device))
                return (bce(out.squeeze(1), tt) * w).sum() / v.sum().clamp(min=1)
            info["pos_weight"] = None if pw is None else round(float(pw), 2)
        else:
            cnt = np.bincount(masks[valid].ravel(), minlength=K); ratio = cnt / cnt.sum()
            info["클래스 비율"] = " ".join(f"{r:.1%}" for r in ratio[:3]) + (" ..." if K > 3 else "")
            cw = None
            if class_weight == "auto":
                cw = 1 / np.sqrt(np.maximum(ratio, 1e-6)); cw = torch.tensor(cw / cw.mean(), dtype=torch.float32, device=device)
            ce = nn.CrossEntropyLoss(weight=cw, ignore_index=IGNORE)
            crit = lambda out, t: ce(out, t)
            info["class_weight"] = class_weight
        model = UNet(out_ch=K).to(device); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        tl = DataLoader(tr, batch_size=16, shuffle=True); vl = DataLoader(te, batch_size=16)
        def evaluate():
            model.eval(); ious, fps = [], []; inter = np.zeros(K); union = np.zeros(K)
            with torch.no_grad():
                for x, y in vl:
                    x, y = x.to(device), y.to(device); out = model(x)
                    if task == "binary":
                        pred = out.squeeze(1) > 0.5
                        for p, t in zip(pred, y):
                            v = t != IGNORE; t1 = (t == 1) & v; p1 = p & v
                            if t1.sum() == 0: fps.append((p1.sum() / v.sum()).item())
                            else: ious.append(((p1 & t1).sum() / (p1 | t1).sum()).item())
                    else:
                        pred = out.argmax(1); v = y != IGNORE
                        for k in range(K):
                            pk, tk = (pred == k) & v, (y == k) & v; inter[k] += (pk & tk).sum().item(); union[k] += (pk | tk).sum().item()
            if task == "binary": return float(np.mean(ious)), (float(np.mean(fps)) if fps else None)
            return float(np.nanmean(np.where(union > 0, inter / np.maximum(union, 1), np.nan))), None
        ious, secs = [], []
        E = 1 if QUICK else epochs
        for ep in range(E):
            model.train(); sync(); t = time.time()
            for x, y in tl:
                x, y = x.to(device), y.to(device); opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            sync(); secs.append(time.time() - t)
            iou, fp = evaluate(); ious.append(iou)
        row = {"설정": name, "학습 장 수": len(tr), "에폭": E, "에폭당 학습 시간(초)": round(float(np.mean(secs)), 1),
               "최종 IoU": round(ious[-1], 3), "최고 IoU(에폭)": f"{max(ious):.3f} ({int(np.argmax(ious)) + 1})",
               "대상 없는 사진 오탐": None if fp is None else f"{fp:.2%}", "장당 추론(ms)": round(infer_ms(model, 128), 2), **info}
        print(row); return row

    rows3 = [run_template("Pet 이진 (반려동물)", "pet", "binary"),
             run_template("Pet 3클래스 + 클래스 가중치", "pet", "multi", class_weight="auto"),
             run_template("VOC 사람, 가중치 없음", "voc", "binary"),
             run_template("VOC 사람, pos_weight auto", "voc", "binary", pos_weight="auto")]
    df3 = pd.DataFrame(rows3); print(f"=== 3부 결과 ({GPU_NAME}) ==="); display(df3)

In [ ]:
# 결과 한 번에 보기 (다운로드 전에 확인)
print("GPU:", GPU_NAME)
for nm in ["df1", "df2", "df3"]:
    if nm in globals():
        print(f"\n### {nm}"); print(globals()[nm].to_string(index=False))